# 005 多证据候选排序 — 可复现入口（E0→E3，不重跑 GPU）

**结论已出，本 notebook 为可复现入口，不重跑 GPU。** 本轮已有结论全部来自本地/CPU复算与已完成的远端试点（见配对进展）；本 notebook 只读已落盘产物做一致性核对，不做神经前向、不做训练、不联网、不读凭据。

统一候选池定义以 `artifacts/005/candidate_manifest.json` 为准（并集 712199 shipped；canon 去重 706242；official 去重 706289；554 组横跨双池；NP16 单列）。各格输入产物与已跑结论见各 cell 首部注释；已落盘代码入口：
- `src/molecule_ranker.py`（E2 分子级 ranker 落盘重跑）
- `src/forward_rerank.py`（E3 GLACIER 条件映射/缓存/预算/融合）
- `scripts/mist_cf_formula.py`（E4 MIST-CF 公式重排落盘入口，本 notebook E3 格附带引用，不重跑）

运行顺序：身份映射 → E0 召回 → E1 检索 → E2 ranker → E3 forward。全部只读 CPU 秒级。


In [ ]:
"""身份映射格：统一并集候选池核对（只读，不重跑 GPU）。
输入产物：artifacts/005/candidate_manifest.json（本格 authoritative）、artifacts/005/structure_identity_summary.json、artifacts/005/structure_identity_train.parquet、artifacts/005/structure_identity_coco.parquet。
已跑结论：train 275810行/canon 274186/变更4370；coco 436389行/canon 432610/变更45540；并集shipped 712199、canon去重706242、official去重706289、零shipped重叠、554组横跨双池、NP16单列；caliber=rdkit_full，本地RDKit 2026.03.6，官方口径2026.03.3（偏差已记）。
结论已出，本 notebook 为可复现入口，不重跑 GPU。
"""
import json

manifest = json.load(open("artifacts/005/candidate_manifest.json"))
summary = json.load(open("artifacts/005/structure_identity_summary.json"))
u = manifest["union_pool"]
print("union shipped:", u["n_shipped"])
print("union canon:", u["n_unique_canonical"])
print("union official:", u["n_unique_official"])
print("cross groups:", u["cross_pool_canonical_groups_554"])
print("NP16:", u["np250_changed_16_count"])
assert u["n_shipped"] == 712199
assert u["n_unique_canonical"] == 706242
assert u["n_unique_official"] == 706289
assert u["cross_pool_canonical_groups_554"] == 554
assert u["np250_changed_16_count"] == 16
assert summary["union"]["n_unique_canonical"] == 706242
assert summary["union"]["n_unique_official"] == 706289
print("fp_bits sha:", manifest["fp_bits"]["sha256"])
assert manifest["fp_bits"]["sha256"].startswith("a61e37")
print("rdkit local/official:", manifest["rdkit"]["local_measured"], manifest["rdkit"]["official_caliber"])
print("OK: 身份映射与manifest一致（只读核对）")


In [ ]:
"""E0 格：同池复现与身份修正核对（只读，不重跑 GPU）。
输入产物：artifacts/005/e0_cq_recall.json（36组池定义）、artifacts/005/e0_cq_windows.npz（3000键=12前缀×250）、artifacts/005/e0_cq_selected.json、artifacts/005/e0_canon_train.json、artifacts/005/e0_canon_union.json、artifacts/005/e0_official_union.json、artifacts/005/e0_head_compare.json、artifacts/005/e0_logits_meta.json。
已跑结论：train shipped orig 0.675203逐位复现历史R2；canon重算train orig 0.688468 / union orig 0.504854；扩池代价orig head下train→union约-0.183（23分子掉出top25）；rank-head union -0.045（53改善/84恶化），train上基本持平；e0_head_compare的summary/detail为空遗留，数字以canon明细+logits为准。
结论已出，本 notebook 为可复现入口，不重跑 GPU。
"""
import json
import numpy as np

recall = json.load(open("artifacts/005/e0_cq_recall.json"))
print("pool_info:", recall["pool_info"])
assert recall["pool_info"] == {"n_train": 275810, "n_coco": 436389, "n_coco_extra": 436389, "n_union": 712199}
print("combos:", len(recall["combos"]))
assert len(recall["combos"]) == 36
print("query_def:", recall["query_def"])
w = np.load("artifacts/005/e0_cq_windows.npz", allow_pickle=True)
print("windows keys:", len(w.files))
assert len(w.files) == 3000
ct = json.load(open("artifacts/005/e0_canon_train.json"))
cu = json.load(open("artifacts/005/e0_canon_union.json"))
ou = json.load(open("artifacts/005/e0_official_union.json"))
print("canon train orig:", ct["orig"]["MRR@25"], "union orig:", cu["mrr"]["orig"]["MRR@25"])
print("official union unique:", ou["unique_official"])
assert ou["unique_official"] == 706289
assert cu["union_stats"]["unique_canon"] == 706242
meta = json.load(open("artifacts/005/e0_logits_meta.json"))
print("logits n_spectra:", meta["n_spectra"], "orig sha:", meta["orig_weights"]["sha256"][:12])
assert meta["n_spectra"] == 1184
print("OK: E0召回与canon/official口径一致（只读核对）")


In [ ]:
"""E1 格：完整背景谱库 + analog 核对（只读，不重跑 GPU）。
输入产物：src/retrieval.py（+697行，直接/偏移disjoint接口）、artifacts/005/e2_feature_schema.json（direct-only声明）、artifacts/005/e2_metrics.json（lib_only对照）。
已跑结论：完整背景seen谱MRR 0.9263/分子0.9228（相对旧L2453约-0.049/-0.056，Top25同为1179/1184，召回同、排序变差）；unseen真值证据MRR 0.0（tie-break 0.136已剔除，不冒充）；谱库修94个FPNet Top1错误、反向8个；高相似误命中121例（含同adduct多源仍错），证阈值不可硬接管；偏移全背景穷举本地未跑（局限如实声明）。
结论已出，本 notebook 为可复现入口，不重跑 GPU。
"""
import json
from src import retrieval as ret

print("retrieval module:", ret.__name__)
assert hasattr(ret, "aggregate_structure_hits") or hasattr(ret, "spectral_entropy") or hasattr(ret, "ce_median")
schema = json.load(open("artifacts/005/e2_feature_schema.json"))
print("schema Cq:", schema["Cq"])
print("schema notes:", schema["notes"])
assert "direct only" in schema["notes"]
metrics = json.load(open("artifacts/005/e2_metrics.json"))["metrics"]
print("lib_only MRR:", metrics["lib_only"]["mrr@25"], "single MRR:", metrics["single"]["mrr@25"])
assert abs(metrics["lib_only"]["mrr@25"] - 0.8932555999556) < 1e-9
print("OK: E1检索接口与lib对照一致（只读核对，不跑全背景穷举）")


In [ ]:
"""E2 格：分子级多证据 ranker 核对（只读，不重跑 GPU）。
输入产物：src/molecule_ranker.py（落盘重跑：C(q)构造/FP点积/OOF/排序评估）、artifacts/005/e2_metrics.json、artifacts/005/e2_oof_predictions.json、artifacts/005/e2_per_molecule_top25.json、artifacts/005/e2_feature_schema.json；复用artifacts/005/e0_logits_single_s2_np1184.npy（不重跑前向）。
已跑结论：同一C(q)（union|single|10ppm nocap）single 0.5149、等权0.4956、选谱权重0.5112、ranker 0.9211；lib_only 0.8933；unseen真值去库模拟0.1000；主审独立重算1000个rank声明0错位、ranker MRR分毫不差；verdict不进提交管线（+0.406几乎全来自谱库，ranker仅再+0.028，无库真值下系统性降权）。
结论已出，本 notebook 为可复现入口，不重跑 GPU。
"""
import json
import src.molecule_ranker as mr

print("E2_FEATURES:", len(mr.E2_FEATURES), mr.E2_FEATURES[:3])
assert mr.E2_FEATURES == json.load(open("artifacts/005/e2_feature_schema.json"))["features"]
assert mr.E2_CQ_LABEL == "union|single|10.0"
assert mr.E2_SEED == 20261004 and mr.E2_K == 5
metrics = json.load(open("artifacts/005/e2_metrics.json"))["metrics"]
print("single:", metrics["single"]["mrr@25"], "ranker:", metrics["ranker"]["mrr@25"])
assert abs(metrics["ranker"]["mrr@25"] - 0.921142857142857) < 1e-9
assert abs(metrics["single"]["mrr@25"] - 0.5149267618854575) < 1e-9
rep = mr.verify_saved_oof()
print("rerun MRR:", rep["metrics_rerun"]["mrr@25"], "diff:", rep["mrr_abs_diff"], "ok:", rep["ok"])
assert rep["ok"] and rep["mrr_abs_diff"] <= 1e-6
print("OK: E2 ranker回放一致，verdict不进提交管线（只读核对）")


In [ ]:
"""E3 格：GLACIER 同式重排 + 自适应预算核对（只读，不重跑 GPU）。
输入产物：src/forward_rerank.py（条件映射/五元组缓存键/四类计数/三种预算入口/同式slot-preserving融合）、artifacts/005/glacier.pt（sha 9192e4c2…）、artifacts/005/e3_np250_fusion_eval.json（base 0.5146/hard 0.6141/z-融合0.6391）、artifacts/005/e3_pilot_rerank_eval.json、artifacts/005/np250_shortlists_top50.json；公式路径另见 scripts/mist_cf_formula.py + artifacts/005/mist_cf_formula_topk_20.json（20分子质量基线Top1 9/20→MIST重排15/20，SIRIUS缺席已声明，本格只引用不重跑）。
已跑结论：28分子试点base 0.5162→GLACIER 0.5791（+0.0629，改善8/恶化6/持平14）；NP250全量base 0.5146/hard 0.6141(+88/-49)/z-融合0.6391(+91/-25)，融合>硬替换>base，保住24个硬替换会亏的分子；只融合不硬覆盖；NP250为反复使用的开发集，0.6391不是成绩承诺。
结论已出，本 notebook 为可复现入口，不重跑 GPU。
"""
import json
import src.forward_rerank as fr

print("covered adducts:", fr.COVERED_ADDUCTS, "model:", fr.MODEL_VERSION[:24])
assert "[M+H]+" in fr.COVERED_ADDUCTS
e3 = json.load(open("artifacts/005/e3_np250_fusion_eval.json"))
print("e3 n/base/hard/fused:", e3["n"], round(e3["base"], 4), round(e3["hard"], 4), round(e3["fused"], 4))
assert e3["n"] == 250
assert abs(e3["base"] - 0.5145934285521241) < 1e-9
assert abs(e3["fused"] - 0.6390605932544295) < 1e-9
mist = json.load(open("artifacts/005/mist_cf_formula_topk_20.json"))
print("mist summary:", mist["summary"])
assert mist["summary"] == {"n": 20, "mass_top1": 9, "mist_top1": 15}
print("OK: E3融合与公式小试引用一致（只读核对）")
